In [1]:
import os
import requests
import smtplib
import asyncio

In [2]:
from dotenv import load_dotenv
from email.message import EmailMessage

In [3]:
from agents import Agent, Runner, trace, function_tool, ModelSettings
from agents.extensions.visualization import draw_graph

In [4]:
load_dotenv(override=True)

EMAIL_ADDRESS = os.getenv("EMAIL_ADDRESS")
EMAIL_SMTP_SERVER = os.getenv("EMAIL_SMTP_SERVER")
EMAIL_APP_PASSWORD = os.getenv("EMAIL_APP_PASSWORD")

MODEL_NAME = "gpt-5.4-mini"

In [5]:
def send_email(subject, text_body, html_body):
    msg = EmailMessage()
    msg["From"] = EMAIL_ADDRESS
    msg["To"] = EMAIL_ADDRESS
    msg["Subject"] = subject
    msg.set_content(text_body)
    msg.add_alternative(html_body, subtype="html")

    with smtplib.SMTP(EMAIL_SMTP_SERVER, 587) as server:
        server.starttls()
        server.login(EMAIL_ADDRESS, EMAIL_APP_PASSWORD)
        server.send_message(msg)

In [6]:
intro = """
You are a sales agent working for Taxee, 
a company that provide SaaS tool for ensuring, tax planning and tax filing process both 
for individuals and businesses, powered by AI.
You write emails.
"""

instruction1 = intro + "Your email style is professional, serious, with gravitas and credibility."
instruction2 = intro + "Your email style is witty, engaging, and humorous."
instruction3 = intro + "Your email style is concise, to the point, in the style of a busy senior executive."

In [7]:
sales_agent1 = Agent("Professional Agent", instructions=instruction1, model=MODEL_NAME)
sales_agent2 = Agent("Witty Agent", instructions=instruction2, model=MODEL_NAME)
sales_agent3 = Agent("Executive Agent", instructions=instruction3, model=MODEL_NAME)

In [8]:
# decision = """
# You pick the best cold email from thr given options.
# Imagine you are a customer and pick the one you are most likely to respond to.
# Do not give an explanation; reply with the selected email only.
# """

# sales_picker = Agent("Sales Picker", instructions=decision, model=MODEL_NAME)

In [9]:
@function_tool
def send_email_tool(subject: str, text_body: str, html_body: str):
    """Send out an email with given subject and body to all sales prospects
    
    Args:
        subject: The subject of the email
        text_body: The body of the email in plain text
        html_body: The HTML body of the email
    """
    send_email(subject, text_body, html_body)
    return "Email sent successully..."

In [10]:
decision = """
You pick the best cold email from thr given options.
Imagine you are a customer and pick the one you are most likely to respond to.
Then use your tool to send the email.
"""
require_tool = ModelSettings(tool_choice="required")
sales_picker = Agent("Sales Picker", instructions=decision, model=MODEL_NAME, 
                     tools=[send_email_tool], model_settings=require_tool)

In [11]:
# message = "Write a cold email"

# with trace("Parellel cold emails"):
#     results = await asyncio.gather(
#         Runner.run(sales_agent1, message),
#         Runner.run(sales_agent2, message),
#         Runner.run(sales_agent3, message)
#     )
    
# outputs = [result.final_output for result in results]

# for output in outputs:
#     print(output + "\n\n")

In [12]:
# message = "Write a cold email"

# with trace("Email selection workflow"):
#     results = await asyncio.gather(
#         Runner.run(sales_agent1, message),
#         Runner.run(sales_agent2, message),
#         Runner.run(sales_agent3, message)
#     )
    
#     outputs = [result.final_output for result in results]
#     emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)
#     best_email = await Runner.run(sales_picker, emails)
#     print(f"Best Email: \n{best_email.final_output}")

In [13]:
message = "Write a cold email"

with trace("Email selection workflow with sending"):
    results = await asyncio.gather(
        Runner.run(sales_agent1, message),
        Runner.run(sales_agent2, message),
        Runner.run(sales_agent3, message)
    )
    
    outputs = [result.final_output for result in results]
    emails = "Cold sales emails:\n\n" + "\n\nEmail:\n\n".join(outputs)
    response = await Runner.run(sales_picker, emails)
    print(f"Final response: \n{response.final_output}")

Final response: 
Chosen and sent: **“Taxes, but make them less terrifying”**

It felt the most likely to get a response because it’s more human, attention-grabbing, and less generic than the others.


In [23]:
import sys

# 1. Force the path to the VERY FRONT of Python's search sequence
graphviz_path = r"D:\My Work\Udemy\AI - new Ed Donner\llm_engineering\.venv\Lib\site-packages\pygments\lexers"
os.environ["PATH"] = graphviz_path + os.pathsep + os.environ["PATH"]

# 2. Double check if the executable actually exists at that path
exe_check = os.path.exists(os.path.join(graphviz_path, "dot.exe"))
print(f"Is dot.exe actually in that folder? {exe_check}")

Is dot.exe actually in that folder? False


In [20]:
draw_graph(sales_picker)

ExecutableNotFound: failed to execute WindowsPath('dot'), make sure the Graphviz executables are on your systems' PATH